# Ternary NoProp — Scale Runs (5 k / 50 k steps)

Companion to `noprop_ternary_colab_free_tier.ipynb`.
Use this notebook **after Wave 1 smoke tests pass** to run full-scale validation.

---

## When to use this notebook
Run `noprop_ternary_colab_free_tier.ipynb` first (800-step smoke).
Come here once ≥ 1 Wave-1 variant passes the loss/dead-rate gates.

---

## Platform Timing Reference — Scale Runs

### 5 k-step validation run (7 experiments total)

| Platform | Accelerator | 3 Global exps | 4 NoProp exps | **Total** | Fits session? |
|---|---|---|---|---|---|
| Google Colab Free | T4 15 GB | ~19 min | ~4.0 h | **~4.3 h** | Marginal |
| Google Colab Pro | A100 40 GB | ~7 min | ~1.5 h | **~1.6 h** | ✅ Yes |
| Kaggle Free | P100 16 GB | ~20 min | ~3.8 h | **~4.1 h** | ✅ Yes (12 h limit) |
| Kaggle Free | T4 15 GB | ~19 min | ~4.0 h | **~4.3 h** | ✅ Yes (12 h limit) |
| SageMaker Studio Lab | T4 15 GB | ~19 min | ~4.0 h | **~4.3 h** | Marginal (4 h GPU) |
| Apple MPS (M1/M2/M3) | — | ~75 min | ~11 h | **~12 h** | Run overnight |

### 50 k-step publication run (best config + 0C baseline — 2 experiments)

| Platform | Accelerator | 0C global | Best NoProp | **Total** | Fits session? |
|---|---|---|---|---|---|
| Google Colab Free | T4 15 GB | ~3.2 h | ~8.5 h | **~11.7 h** | ❌ Tight |
| Google Colab Pro | A100 40 GB | ~1.2 h | ~3.2 h | **~4.4 h** | ✅ Yes |
| Kaggle Free | P100 16 GB | ~3.0 h | ~8.0 h | **~11 h** | ✅ Yes (12 h limit) |
| Lambda Labs A100 SXM4 | — | ~18 min | ~48 min | **~66 min** | ✅ Yes |

> **Recommended for 50 k:** Kaggle Free P100 (12 h session, persistent `/kaggle/working/` storage).


In [ ]:
# ── Colab / Kaggle / Studio Lab setup ───────────────────────────────────────────
import os, sys, json, time, random, math, platform
from pathlib import Path

IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

import subprocess
subprocess.run(['pip', 'install', '-q', '-U', 'torch', 'datasets', 'transformers',
                'pandas', 'matplotlib', 'tqdm'], check=True)


In [ ]:
# ── Scale-run budget selector ────────────────────────────────────────────────────
# Edit these two constants to control run length.
# 5_000 = validation run  |  50_000 = publication run

STEPS_GLOBAL = 5_000    # for the 3 global baseline experiments
STEPS_NOPROP = 5_000    # for the NoProp ternary experiments

# ─ For publication runs, uncomment these and pick ONE best NoProp mode ──────────
# STEPS_GLOBAL = 50_000
# STEPS_NOPROP = 50_000

# ─ Select which NoProp modes to run (set to [] to skip all NoProp) ───────────────
# Remove modes that failed Wave 1 smoke gates to save time.
NOPROP_MODES = ['ste', 'trust', 'dqt', 'hestia']   # ← remove failing modes here

print(f'Global steps : {STEPS_GLOBAL:,}')
print(f'NoProp steps : {STEPS_NOPROP:,}')
print(f'NoProp modes : {NOPROP_MODES}')
print()
# Wave-0 baseline table
print('Wave 0 experiments (baselines):')
print(f'  {"ID":<28} {"Kind":>8} {"Mode":>8} {"Steps":>8}')
print('  ' + '─'*55)
_w0 = [
    ('0A_fp32_noprop',      'noprop', 'fp32',  STEPS_NOPROP),
    ('0B_ternary_backprop', 'global', 'trust', STEPS_GLOBAL),
    ('0C_fp32_backprop',    'global', 'fp32',  STEPS_GLOBAL),
]
for _id, _k, _m, _s in _w0:
    print(f'  {_id:<28} {_k:>8} {_m:>8} {_s:>8,}')
print()
# Wave-1 existence table
print('Wave 1 experiments (NoProp × ternary):')
print(f'  {"ID":<28} {"Kind":>8} {"Mode":>8} {"Steps":>8}')
print('  ' + '─'*55)
_w1 = [(f'1{chr(65+i)}_noprop_{m}', 'noprop', m, STEPS_NOPROP)
       for i, m in enumerate(NOPROP_MODES)]
for _id, _k, _m, _s in _w1:
    print(f'  {_id:<28} {_k:>8} {_m:>8} {_s:>8,}')


In [ ]:
# ── Paths, seed, hardware probe ─────────────────────────────────────────────────
import torch
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

RUN_NAME = time.strftime('scale_%Y%m%d_%H%M%S')
if IN_COLAB:
    OUT_DIR = Path('/content/drive/MyDrive/drex_noprop_ternary_scale')
elif IN_KAGGLE:
    OUT_DIR = Path('/kaggle/working/drex_noprop_ternary_scale')
else:
    OUT_DIR = Path('/tmp/drex_noprop_ternary_scale')

RUN_DIR  = OUT_DIR / RUN_NAME
CKPT_DIR = RUN_DIR / 'checkpoints'
for _d in [RUN_DIR, CKPT_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device  :', device)
if torch.cuda.is_available():
    _p = torch.cuda.get_device_properties(0)
    print('GPU     :', _p.name, f'({_p.total_memory/1024**3:.1f} GB)')
print('Python  :', platform.python_version())
print('Torch   :', torch.__version__)
print('Run dir :', RUN_DIR)


In [ ]:
# ── Dataset (WikiText-2) ─────────────────────────────────────────────────────────
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

tokenizer = AutoTokenizer.from_pretrained('gpt2')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

class LMDataset(Dataset):
    def __init__(self, token_ids, seq_len=128):
        self.seq_len = seq_len
        self.tokens  = token_ids
        self.n       = max(0, (len(self.tokens) - 1) // self.seq_len)
    def __len__(self): return self.n
    def __getitem__(self, idx):
        i = idx * self.seq_len
        return (torch.tensor(self.tokens[i:i+self.seq_len],   dtype=torch.long),
                torch.tensor(self.tokens[i+1:i+self.seq_len+1], dtype=torch.long))

def build_wikitext2(seq_len=128, batch_size=16):
    _tr = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
    _va = load_dataset('wikitext', 'wikitext-2-raw-v1', split='validation')
    _tok = lambda txt: tokenizer('\n'.join(txt['text']), add_special_tokens=False)['input_ids']
    _td = LMDataset(_tok(_tr), seq_len)
    _vd = LMDataset(_tok(_va), seq_len)
    return (DataLoader(_td, batch_size=batch_size, shuffle=True,  drop_last=True),
            DataLoader(_vd, batch_size=batch_size, shuffle=False, drop_last=False))

train_loader, val_loader = build_wikitext2()
print('Train batches:', len(train_loader), ' Val batches:', len(val_loader))


In [ ]:
# ── Quantization primitives ──────────────────────────────────────────────────────
class TernarySTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, w, delta):
        ctx.save_for_backward(w, delta)
        return torch.where(w > delta, torch.ones_like(w),
               torch.where(w < -delta, -torch.ones_like(w), torch.zeros_like(w)))
    @staticmethod
    def backward(ctx, g):
        w, delta = ctx.saved_tensors
        return g * (w.abs() <= 1.5 * delta).to(g.dtype), None

def ternary_ste(w, delta): return TernarySTE.apply(w, delta)
def trust_scale(w, d): return (1.0 / (1.0 + (w.abs() - d).abs())).clamp(0.05, 1.0)
def stochastic_ternary_project(w, d):
    p = (w.abs() / (d + 1e-8)).clamp(0.0, 1.0)
    return (torch.rand_like(w) < p).float() * w.sign()

class BitLinear(nn.Module):
    def __init__(self, in_f, out_f, mode='fp32', bias=True):
        super().__init__()
        self.mode = mode
        if mode == 'hestia':
            self.logits = nn.Parameter(torch.randn(out_f, in_f, 3) * 0.02)
        else:
            self.weight = nn.Parameter(torch.randn(out_f, in_f) * 0.02)
        self.bias = nn.Parameter(torch.zeros(out_f)) if bias else None
    def _delta(self, w): return 0.7 * w.abs().mean().detach() + 1e-8
    def forward(self, x, tau=1.0):
        if   self.mode == 'fp32':   wq = self.weight
        elif self.mode == 'ste':    wq = ternary_ste(self.weight, self._delta(self.weight))
        elif self.mode == 'trust':
            d = self._delta(self.weight)
            wq = ternary_ste(self.weight, d) * trust_scale(self.weight, d)
        elif self.mode == 'dqt':    wq = stochastic_ternary_project(self.weight, self._delta(self.weight))
        elif self.mode == 'hestia':
            states = torch.tensor([-1.,0.,1.], device=x.device).view(1,1,3)
            wq = (F.softmax(self.logits / max(tau, 1e-3), dim=-1) * states).sum(-1)
        else: raise ValueError(f'Unknown mode: {self.mode}')
        return F.linear(x, wq, self.bias)


In [ ]:
# ── Model ────────────────────────────────────────────────────────────────────────
class TinyBlock(nn.Module):
    def __init__(self, d_model, n_heads, mode='fp32'):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln1  = nn.LayerNorm(d_model)
        self.ff1  = BitLinear(d_model, 4*d_model, mode=mode)
        self.ff2  = BitLinear(4*d_model, d_model, mode=mode)
        self.ln2  = nn.LayerNorm(d_model)
    def forward(self, x, tau=1.0):
        T = x.size(1)
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        a, _ = self.attn(x, x, x, attn_mask=mask)
        x = self.ln1(x + a)
        return self.ln2(x + self.ff2(F.gelu(self.ff1(x, tau=tau)), tau=tau))

class TinyNoPropTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=4, n_blocks=6, mode='fp32', max_len=256):
        super().__init__()
        self.mode = mode; self.n_blocks = n_blocks
        self.tok  = nn.Embedding(vocab_size, d_model)
        self.pos  = nn.Embedding(max_len, d_model)
        self.blocks     = nn.ModuleList([TinyBlock(d_model, n_heads, mode) for _ in range(n_blocks)])
        self.noise_proj = nn.ModuleList([BitLinear(d_model, d_model, mode='fp32') for _ in range(n_blocks)])
        self.head = BitLinear(d_model, vocab_size, mode='fp32')
    def base_embed(self, x):
        B, T = x.shape
        return self.tok(x) + self.pos(torch.arange(T, device=x.device).unsqueeze(0).expand(B, T))
    def forward_global(self, x, tau=1.0):
        h = self.base_embed(x)
        for blk in self.blocks: h = blk(h, tau=tau)
        return self.head(h)
    def forward_block_local(self, b, x, z_noisy, tau=1.0):
        h = self.base_embed(x).detach() + self.noise_proj[b](z_noisy.detach())
        return self.blocks[b](h, tau=tau), self.head(self.blocks[b](h, tau=tau))

def dead_neuron_rate(t): return (t.abs() < 1e-6).float().mean().item()


In [ ]:
# ── Utilities ────────────────────────────────────────────────────────────────────
def evaluate_ppl(model, val_loader, max_batches=100):
    model.eval()
    losses = []
    with torch.no_grad():
        for i, (x, y) in enumerate(val_loader):
            if i >= max_batches: break
            x, y = x.to(device), y.to(device)
            loss = F.cross_entropy(model.forward_global(x).reshape(-1, model.head.weight.shape[0] if hasattr(model.head, 'weight') else len(tokenizer)), y.reshape(-1))
            losses.append(loss.item())
    model.train()
    m = sum(losses) / max(1, len(losses))
    return m, math.exp(min(m, 20.0))

def get_grad_norm(module):
    sq = sum(float((p.grad.detach()**2).sum()) for p in module.parameters() if p.grad is not None)
    return math.sqrt(max(sq, 0.0))

def save_ckpt(model, opt, step, tag):
    path = CKPT_DIR / f'{tag}_step_{step:07d}.pt'
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict() if not isinstance(opt, list) else [o.state_dict() for o in opt], 'step': step}, path)
    return str(path)


In [ ]:
# ── Runners ──────────────────────────────────────────────────────────────────────
def run_global(exp_id, mode='fp32', steps=5000, lr=3e-4, eval_every=500):
    vocab  = len(tokenizer)
    model  = TinyNoPropTransformer(vocab_size=vocab, mode=mode).to(device)
    opt    = torch.optim.AdamW(model.parameters(), lr=lr)
    logs, it, tau = [], iter(train_loader), 1.5
    for step in tqdm(range(1, steps+1), desc=exp_id):
        try: x, y = next(it)
        except StopIteration: it = iter(train_loader); x, y = next(it)
        x, y = x.to(device), y.to(device)
        opt.zero_grad(set_to_none=True)
        loss = F.cross_entropy(model.forward_global(x, tau=tau).reshape(-1, vocab), y.reshape(-1))
        loss.backward(); gn = get_grad_norm(model); opt.step()
        if mode == 'hestia': tau = max(0.05, tau * 0.999)
        if step % eval_every == 0 or step == 1:
            vl, vp = evaluate_ppl(model, val_loader)
            mem = torch.cuda.max_memory_allocated()/1024**3 if torch.cuda.is_available() else 0.
            logs.append({'exp_id':exp_id,'step':step,'train_loss':float(loss),'val_loss':vl,'val_ppl':vp,'grad_norm':gn,'dead_rate':0.,'max_mem_gb':mem})
    return model, pd.DataFrame(logs), save_ckpt(model, opt, steps, exp_id)

def run_noprop(exp_id, mode='ste', steps=5000, lr=3e-4, eval_every=500):
    vocab  = len(tokenizer)
    model  = TinyNoPropTransformer(vocab_size=vocab, mode=mode, n_blocks=6).to(device)
    shared = list(model.tok.parameters()) + list(model.pos.parameters()) + list(model.head.parameters())
    bopts  = [torch.optim.AdamW(list(model.blocks[b].parameters()) + list(model.noise_proj[b].parameters()) + shared, lr=lr) for b in range(model.n_blocks)]
    logs, it, tau   = [], iter(train_loader), 1.5
    noise_sched     = torch.linspace(1.0, 0.1, model.n_blocks, device=device)
    for step in tqdm(range(1, steps+1), desc=exp_id):
        try: x, y = next(it)
        except StopIteration: it = iter(train_loader); x, y = next(it)
        x, y = x.to(device), y.to(device)
        y_emb = model.tok(y).detach()
        gnorms, drates, blosses = [], [], []
        for b in range(model.n_blocks):
            bopts[b].zero_grad(set_to_none=True)
            z = y_emb + noise_sched[b] * torch.randn_like(y_emb)
            out, logits = model.forward_block_local(b, x, z, tau=tau)
            alpha_t = 1.0 + 0.1 * (b / max(1, model.n_blocks-1))
            loss = alpha_t * F.mse_loss(out, y_emb) + 0.1 * F.cross_entropy(logits.reshape(-1, vocab), y.reshape(-1))
            loss.backward()
            gnorms.append(get_grad_norm(model.blocks[b]))
            drates.append(dead_neuron_rate(out.detach()))
            blosses.append(float(loss))
            bopts[b].step()
        if mode == 'hestia': tau = max(0.05, tau * 0.999)
        if step % eval_every == 0 or step == 1:
            vl, vp = evaluate_ppl(model, val_loader)
            mem = torch.cuda.max_memory_allocated()/1024**3 if torch.cuda.is_available() else 0.
            logs.append({'exp_id':exp_id,'step':step,'train_loss':sum(blosses)/len(blosses),'val_loss':vl,'val_ppl':vp,'grad_norm':sum(gnorms)/len(gnorms),'dead_rate':sum(drates)/len(drates),'max_mem_gb':mem})
    return model, pd.DataFrame(logs), save_ckpt(model, bopts[0], steps, exp_id)


In [ ]:
# ── Build experiment list from selectors above ───────────────────────────────────
EXPERIMENTS_SCALE = [
    {'id':'0A_fp32_noprop',      'wave':0,'kind':'noprop','mode':'fp32',  'steps':STEPS_NOPROP,
     'desc':'NoProp FP32 baseline','gate_ppl':250,'gate_dead':0.30},
    {'id':'0B_ternary_backprop', 'wave':0,'kind':'global','mode':'trust', 'steps':STEPS_GLOBAL,
     'desc':'Backprop Trust-ternary baseline','gate_ppl':250,'gate_dead':0.15},
    {'id':'0C_fp32_backprop',    'wave':0,'kind':'global','mode':'fp32',  'steps':STEPS_GLOBAL,
     'desc':'Backprop FP32 gold standard','gate_ppl':200,'gate_dead':0.10},
] + [
    {'id':f'1{chr(65+i)}_noprop_{m}','wave':1,'kind':'noprop','mode':m,'steps':STEPS_NOPROP,
     'desc':f'NoProp + {m.upper()} ternary','gate_ppl':300,'gate_dead':0.30}
    for i, m in enumerate(NOPROP_MODES)
]

# ── Timing calibration ────────────────────────────────────────────────────────
import time as _tm

_CALIB_N = 3
_vocab   = len(tokenizer)
_mc      = TinyNoPropTransformer(vocab_size=_vocab, mode='fp32').to(device)
_oc      = torch.optim.AdamW(_mc.parameters(), lr=3e-4)
_xc, _yc = next(iter(train_loader))
_xc, _yc = _xc.to(device), _yc.to(device)

def _sync():
    if torch.cuda.is_available(): torch.cuda.synchronize()

for _ in range(2):
    _oc.zero_grad()
    F.cross_entropy(_mc.forward_global(_xc).reshape(-1,_vocab),_yc.reshape(-1)).backward()
    _oc.step()
_sync()

_t0 = _tm.perf_counter()
for _ in range(_CALIB_N):
    _oc.zero_grad()
    F.cross_entropy(_mc.forward_global(_xc).reshape(-1,_vocab),_yc.reshape(-1)).backward()
    _oc.step()
_sync()
_g_sps = _CALIB_N / (_tm.perf_counter() - _t0)

_boc = [torch.optim.AdamW(list(_mc.blocks[_b].parameters())+list(_mc.noise_proj[_b].parameters())+list(_mc.tok.parameters())+list(_mc.pos.parameters())+list(_mc.head.parameters()),lr=3e-4) for _b in range(_mc.n_blocks)]
_nc  = torch.linspace(1.0, 0.1, _mc.n_blocks, device=device)
_ye  = _mc.tok(_yc).detach()
_t0  = _tm.perf_counter()
for _ in range(_CALIB_N):
    for _b in range(_mc.n_blocks):
        _boc[_b].zero_grad()
        _z = _ye + _nc[_b]*torch.randn_like(_ye)
        _o, _l = _mc.forward_block_local(_b, _xc, _z)
        (F.mse_loss(_o,_ye)+0.1*F.cross_entropy(_l.reshape(-1,_vocab),_yc.reshape(-1))).backward()
        _boc[_b].step()
_sync()
_n_sps = _CALIB_N / (_tm.perf_counter() - _t0)

del _mc, _oc, _boc, _xc, _yc, _ye
_hw = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ('Apple MPS' if hasattr(torch.backends,'mps') and torch.backends.mps.is_available() else 'CPU')

print(f'Hardware: {_hw}')
print(f'Global  : {_g_sps:.2f} steps/sec  ({1000/_g_sps:.0f} ms/step)')
print(f'NoProp  : {_n_sps:.2f} steps/sec  ({1000/_n_sps:.0f} ms/step)')
print()
print(f'{"#":<3} {"Exp ID":<28} {"W":>2} {"Steps":>7}  {"ETA":>12}')
print('─'*60)
_tot = 0.0
for _i, _e in enumerate(EXPERIMENTS_SCALE, 1):
    _sps = _n_sps if _e['kind']=='noprop' else _g_sps
    _sec = _e['steps'] / _sps; _tot += _sec
    _mm, _ss = divmod(int(_sec), 60)
    _hh, _mm = divmod(_mm, 60)
    _eta = f'{_hh}h {_mm:02d}m {_ss:02d}s' if _hh else f'{_mm}m {_ss:02d}s'
    print(f'{_i:<3} {_e["id"]:<28} {_e["wave"]:>2} {_e["steps"]:>7,}  {_eta:>12}')
print('─'*60)
_hh2, _rem = divmod(int(_tot), 3600)
_mm2, _ss2 = divmod(_rem, 60)
_tot_str = f'{_hh2}h {_mm2:02d}m {_ss2:02d}s' if _hh2 else f'{_mm2}m {_ss2:02d}s'
print(f'{"ALL":<3} {"":28} {"":2} {sum(e["steps"] for e in EXPERIMENTS_SCALE):>7,}  {_tot_str:>12}')


In [ ]:
# ── Run loop (scale) ─────────────────────────────────────────────────────────────
all_logs_s = []
results_s  = []

for _e in EXPERIMENTS_SCALE:
    _done = RUN_DIR / f'{_e["id"]}_logs.csv'
    if _done.exists():
        print(f'SKIP  {_e["id"]} (already done)')
        _df = pd.read_csv(_done); all_logs_s.append(_df); results_s.append(_df.iloc[-1].to_dict())
        continue

    _sps  = _n_sps if _e['kind']=='noprop' else _g_sps
    _sec  = _e['steps'] / _sps
    _mm, _ss = divmod(int(_sec), 60); _hh, _mm = divmod(_mm, 60)
    _eta  = f'{_hh}h {_mm:02d}m {_ss:02d}s' if _hh else f'{_mm}m {_ss:02d}s'
    print(f'\n{"─"*70}\nRUN  {_e["id"]}\n     {_e["desc"]}  steps={_e["steps"]:,}  ETA≈{_eta}\n{"─"*70}')

    _t0 = time.time()
    if _e['kind'] == 'global':
        _model, _logs, _ckpt = run_global(_e['id'], mode=_e['mode'], steps=_e['steps'])
    else:
        _model, _logs, _ckpt = run_noprop(_e['id'], mode=_e['mode'], steps=_e['steps'])
    _elapsed = time.time() - _t0

    _logs.to_csv(_done, index=False); all_logs_s.append(_logs)
    _last = _logs.iloc[-1].to_dict()
    _last.update({'elapsed_sec': round(_elapsed,1), 'checkpoint': _ckpt,
                  'gate_ppl_pass': bool(_last['val_ppl'] < _e['gate_ppl']),
                  'gate_dead_pass': bool(_last['dead_rate'] < _e['gate_dead'])})
    results_s.append(_last)
    _g = 'PASS' if _last['gate_ppl_pass'] and _last['gate_dead_pass'] else 'FAIL'
    print(f'DONE  {_elapsed/60:.1f} min | val_ppl={_last["val_ppl"]:.1f} | dead={_last["dead_rate"]:.3f} | {_g}')

_sumdf = pd.DataFrame(results_s).sort_values('val_ppl')
_sumdf.to_csv(RUN_DIR/'summary.csv', index=False)
print(f'\nSummary saved → {RUN_DIR}/summary.csv')
_sumdf[['exp_id','val_ppl','dead_rate','elapsed_sec','gate_ppl_pass','gate_dead_pass']]


In [ ]:
# ── Plots ─────────────────────────────────────────────────────────────────────────
if all_logs_s:
    _df = pd.concat(all_logs_s, ignore_index=True)
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for _eid, _g in _df.groupby('exp_id'):
        axes[0].plot(_g['step'], _g['val_ppl'],   label=_eid)
        axes[1].plot(_g['step'], _g['dead_rate'],  label=_eid)
        axes[2].plot(_g['step'], _g['grad_norm'],  label=_eid)
    axes[0].set_title('Validation Perplexity')
    axes[1].set_title('Dead Neuron Rate')
    axes[2].set_title('Gradient Norm')
    for _ax in axes:
        _ax.set_xlabel('step'); _ax.legend(fontsize=7); _ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(RUN_DIR/'plots.png', dpi=120); plt.show()
    print('Plot saved:', RUN_DIR/'plots.png')


In [ ]:
# ── Persist JSON run report ───────────────────────────────────────────────────────
_report = {
    'run_name': RUN_NAME,
    'device': str(device),
    'seed': SEED,
    'steps_global': STEPS_GLOBAL,
    'steps_noprop': STEPS_NOPROP,
    'noprop_modes': NOPROP_MODES,
    'run_dir': str(RUN_DIR),
    'experiments': results_s,
}
with open(RUN_DIR/'run_report.json', 'w') as _f:
    json.dump(_report, _f, indent=2)
print('Report saved:', RUN_DIR/'run_report.json')
